In [ ]:
from time import sleep
from json import dumps
from kafka3 import KafkaProducer
import pandas as pd
import datetime as dt

In [ ]:
hostip = "172.23.224.1"

def publish_message(producer_instance, topic_name, data):
    try:
        producer_instance.send(topic_name, value=data)
        print('Message published successfully. ' + str(data))
    except Exception as ex:
        print('Exception in publishing message.')
        print(str(ex))
        

In [ ]:
def connect_kafka_producer():
    _producer = None
    try:
        _producer = KafkaProducer(bootstrap_servers=[f'{hostip}:9092'],
                                  value_serializer=lambda x:dumps(x).encode('utf-8'),
                                  api_version=(0, 10))
    except Exception as ex:
        print('Exception while connecting Kafka.')
        print(str(ex))
    finally:
        return _producer
    

In [ ]:
if __name__ == '__main__':
    topic_name = 'camera_events'
    csv_file = 'data/camera_event_A.csv'
    batch_interval = 5
    producer_id = "A"
    
    print('Publishing records..')
    producer = connect_kafka_producer()
    
    # Load csv using pandas
    df = pd.read_csv(csv_file)
    
    # Get unique batch IDs
    batch_id_list = sorted(df['batch_id'].unique())
    
    for batch_id in batch_id_list:
        batch_df = df[df['batch_id'] == batch_id].copy()
        batch_df['producer_id'] = producer_id

        for _, row  in batch_df.iterrows():
            batch_data = row.to_dict()
            publish_message(producer, topic_name, batch_data)
            
        sleep(batch_interval)